<a href="https://colab.research.google.com/github/gileshall/axonet/blob/main/notebooks/axonet_vae_embedding_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AxoNet UNet VAE: SWC → Embedding Demo

End-to-end example: take a neuron morphology in SWC format, render it from several camera viewpoints, and produce a 128-dimensional embedding vector using the published HuggingFace UNet VAE checkpoint.

**What you will see:**

1. Install `axonet` and grab a couple of sample SWC files from the Allen Institute Patch-seq release.
2. Render each SWC to a 512×512 depth image (the same input format the VAE was trained on).
3. Download the trained VAE checkpoint from HuggingFace and run inference.
4. Pool the latent across multiple poses with several different merge strategies.
5. Compare two neurons graphically — cosine-similarity matrix, PCA scatter, segmentation sanity check.

**Resources**

- Source: <https://github.com/gileshall/axonet>
- Model: <https://huggingface.co/broadinstitute/axonet-vae-stage1>
- Sample data: Tolias-lab mouse motor cortex Patch-seq morphologies, <https://knowledge.brain-map.org/data/FTX8VQ3E93P26HK79V3>

**Runtime:** CPU works (the model is ~60 MB). For faster rendering and inference set `Runtime > Change runtime type > T4 GPU` in Colab.

## 1. Install

Headless OpenGL libraries are required for `moderngl` to rasterize the neuron meshes. On Colab they are installed at the system level; on a local Jupyter kernel they should already be present (or use the Dockerfile in this repo).

Installing `axonet` from GitHub pulls torch, numpy, moderngl, trimesh, pytorch-lightning, and matplotlib transitively.

In [ ]:
# System libs for headless OpenGL (EGL backend used by moderngl).  Colab only.
!apt-get -qq install libegl1-mesa-dev libgles2-mesa-dev libosmesa6 > /dev/null 2>&1

# axonet package (provides the model class, SWC parser, and renderer)
!pip install -q huggingface_hub "axonet @ git+https://github.com/gileshall/axonet.git"

import axonet
print(f"axonet imported from {axonet.__file__}")

## 2. Imports and device

In [ ]:
import os
import random
import urllib.request
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download

from axonet.models.d3_swc_vae import SegVAE2D
from axonet.training.vae_evaluator import render_swc_to_input
from axonet.training.sampling import fibonacci_sphere
from axonet.training.dataset_generator import place_camera_on_sphere
from axonet.visualization.render import NeuroRenderCore, OffscreenContext

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## 3. Fetch sample SWC files

Two Patch-seq morphologies from the Brain Image Library mirror, picked from different transcriptomic classes (excitatory and inhibitory) so the embeddings should be visibly different.

In [ ]:
SWC_BASE = "https://download.brainimagelibrary.org/biccn/zeng/tolias/pseq/morph/2020Q1"
SWC_URLS = {
    "A_excitatory": f"{SWC_BASE}/excitatory/20171204_sample_2.SWC",
    "B_inhibitory": f"{SWC_BASE}/inhibitory/20180213_sample_5.SWC",
}

swc_paths: dict[str, Path] = {}
for name, url in SWC_URLS.items():
    path = Path(f"{name}.swc")
    if not path.exists():
        print(f"Downloading {url}")
        urllib.request.urlretrieve(url, path)
    swc_paths[name] = path
    print(f"  {name}: {path} ({path.stat().st_size:,} bytes)")

## 4. Render a single SWC

`render_swc_to_input` is the exact rasterizer used during VAE training: orthographic projection, supersampled depth map normalized to `[0, 1]`. Output shape is `(H, W) float32`.

In [ ]:
input_img, _seg_target, _camera = render_swc_to_input(swc_paths["A_excitatory"], width=512, height=512)
print(f"Rendered input: shape={input_img.shape}, dtype={input_img.dtype}, range=[{input_img.min():.3f}, {input_img.max():.3f}]")

plt.figure(figsize=(4, 4))
plt.imshow(input_img, cmap="gray")
plt.title("Neuron A — single-pose depth render")
plt.axis("off")
plt.show()

tensor = torch.from_numpy(input_img).float().unsqueeze(0).unsqueeze(0)  # (1, 1, 512, 512)

## 5. Download checkpoint and build model

Architecture matches `config.json` from the HuggingFace repo: 1-channel input, 64 base channels, 6 segmentation classes, 128-channel latent, variational skips. After loading weights the model is set to eval mode.

In [ ]:
HF_REPO = "broadinstitute/axonet-vae-stage1"
ckpt_path = hf_hub_download(repo_id=HF_REPO, filename="pytorch_model.bin")
print(f"Checkpoint: {ckpt_path}")

model = SegVAE2D(
    in_channels=1,
    base_channels=64,
    num_classes=6,
    latent_channels=128,
    skip_mode="variational",
)
model.load_state_dict(torch.load(ckpt_path, map_location=device))
model.to(device).eval()
print(f"Model on {device}; latent_channels=128")

## 6. Single-pose embedding and segmentation

The VAE returns segmentation logits, a depth reconstruction, and a latent `mu` tensor of shape `(B, 128, H/8, W/8)`. Spatially mean-pooling `mu` gives a single 128-d vector per image.

In [ ]:
with torch.no_grad():
    outputs = model(tensor.to(device), return_latent=True)

embedding = outputs["mu"].mean(dim=(2, 3)).cpu()  # (1, 128)
print(f"Embedding shape: {tuple(embedding.shape)}")
print(f"L2 norm:         {embedding.norm().item():.4f}")
print(f"mean / std:      {embedding.mean().item():+.4f} / {embedding.std().item():.4f}")

np.save("embedding.npy", embedding.numpy())

CLASS_NAMES = ["background", "soma", "axon", "basal_dendrite", "apical_dendrite", "other"]
seg_pred = outputs["seg_logits"].argmax(dim=1)[0].cpu().numpy()
print("\nSegmentation class fractions:")
for i, name in enumerate(CLASS_NAMES):
    frac = (seg_pred == i).mean()
    print(f"  {name:>16s}: {frac * 100:6.2f}%")

## 7. Helpers: multi-pose render, encode, merge

A single random viewpoint is noisy. Real downstream use averages over many camera directions so the per-neuron embedding doesn't depend on an arbitrary projection. The training pipeline uses 24 views per neuron (see `axonet/training/dataset_generator.py`); for a demo a handful of fibonacci-sphere directions is plenty.

`MERGE_STRATEGIES` exposes several ways to collapse `(n_views, 128) → (128,)` so you can swap the policy without touching surrounding code.

In [ ]:
def render_poses(swc_path: Path, n_views: int = 8, size: int = 512, seed: int = 0) -> np.ndarray:
    """Render `n_views` orthographic depth maps of one SWC.

    Returns float32 array of shape (n_views, size, size) in [0, 1].
    """
    dirs = fibonacci_sphere(n_views, rng=random.Random(seed))
    out = np.zeros((n_views, size, size), dtype=np.float32)
    with OffscreenContext(size, size, visible=False, samples=0) as ctx:
        core = NeuroRenderCore(ctx)
        core.load_swc(swc_path, segments=18)
        core.set_projection(perspective=False)
        core.color_mode = 1
        core.fit_camera(margin=0.85)
        target = core.camera.target.copy()
        dist = float(np.linalg.norm(core.camera.eye - target))
        for i, d in enumerate(dirs):
            place_camera_on_sphere(core, d, distance=dist)
            depth = core.render_depth(factor=2).astype(np.float32)
            if depth.max() > 0:
                depth /= depth.max()
            out[i] = depth
    return out


def encode_poses(model: SegVAE2D, poses: np.ndarray, device: str, batch_size: int = 8) -> np.ndarray:
    """Encode (N, H, W) depth poses to (N, 128) mu-pooled embeddings."""
    out: list[np.ndarray] = []
    with torch.no_grad():
        for i in range(0, len(poses), batch_size):
            batch = torch.from_numpy(poses[i : i + batch_size]).unsqueeze(1).to(device)
            _z, mu, _logvar, _e2, _e1, _e0 = model.encode(batch)
            out.append(mu.mean(dim=(2, 3)).cpu().numpy())
    return np.concatenate(out, axis=0)


def _l2_normalize(v: np.ndarray, axis: int = -1, eps: float = 1e-8) -> np.ndarray:
    return v / (np.linalg.norm(v, axis=axis, keepdims=True) + eps)


MERGE_STRATEGIES: dict[str, callable] = {
    "mean":            lambda E: E.mean(axis=0),
    "max":             lambda E: E.max(axis=0),
    "l2_then_mean":    lambda E: _l2_normalize(E, axis=1).mean(axis=0),
    "mean_then_l2":    lambda E: _l2_normalize(E.mean(axis=0)[None])[0],
    "median":          lambda E: np.median(E, axis=0),
    "concat_mean_std": lambda E: np.concatenate([E.mean(axis=0), E.std(axis=0)]),
}


def merge(embs: np.ndarray, strategy: str) -> np.ndarray:
    return MERGE_STRATEGIES[strategy](embs)


def cosine(a: np.ndarray, b: np.ndarray) -> float:
    a, b = a.ravel(), b.ravel()
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))

## 8. Render and encode both neurons across many poses

In [ ]:
N_VIEWS = 8

poses_a = render_poses(swc_paths["A_excitatory"], n_views=N_VIEWS, seed=0)
poses_b = render_poses(swc_paths["B_inhibitory"], n_views=N_VIEWS, seed=0)

embs_a = encode_poses(model, poses_a, device)  # (N, 128)
embs_b = encode_poses(model, poses_b, device)

print(f"Per-pose embeddings: A={embs_a.shape}, B={embs_b.shape}")

## 9. Merge strategy comparison

Each row pools the eight per-pose embeddings into a single vector with a different strategy, then reports cosine similarity between the merged vectors for the two neurons. Lower cosine = more distinguishable. Different strategies produce slightly different absolute values, but the *ranking* between neurons is what matters in practice.

In [ ]:
print(f"{'strategy':>20s}  {'dim':>4s}  {'cos(A,B)':>10s}")
for name in MERGE_STRATEGIES:
    a = merge(embs_a, name)
    b = merge(embs_b, name)
    print(f"{name:>20s}  {a.shape[0]:>4d}  {cosine(a, b):>+10.4f}")

## 10. Graphical comparison

Six panels:

- Top row: a sample render of each neuron and a bar plot of the first 32 latent dims (mean over poses).
- Bottom row: the full `2N × 2N` cosine-similarity matrix (red lines mark the A/B boundary), a PCA scatter of the 16 per-pose embeddings, and a numerical summary.

If the model is doing its job, the intra-neuron block means should exceed the inter-neuron block mean.

In [ ]:
all_embs = _l2_normalize(np.concatenate([embs_a, embs_b], axis=0), axis=1)
sim = all_embs @ all_embs.T

centered = all_embs - all_embs.mean(axis=0, keepdims=True)
_u, _s, vt = np.linalg.svd(centered, full_matrices=False)
proj = centered @ vt[:2].T

intra_a = sim[:N_VIEWS, :N_VIEWS][np.triu_indices(N_VIEWS, k=1)].mean()
intra_b = sim[N_VIEWS:, N_VIEWS:][np.triu_indices(N_VIEWS, k=1)].mean()
inter = sim[:N_VIEWS, N_VIEWS:].mean()

fig, axes = plt.subplots(2, 3, figsize=(13, 8))

axes[0, 0].imshow(poses_a[0], cmap="gray")
axes[0, 0].set_title(f"Neuron A pose 0\n{swc_paths['A_excitatory'].name}")
axes[0, 0].axis("off")

axes[0, 1].imshow(poses_b[0], cmap="gray")
axes[0, 1].set_title(f"Neuron B pose 0\n{swc_paths['B_inhibitory'].name}")
axes[0, 1].axis("off")

ax = axes[0, 2]
dims = np.arange(32)
w = 0.4
ax.bar(dims - w / 2, embs_a.mean(0)[:32], width=w, label="A mean")
ax.bar(dims + w / 2, embs_b.mean(0)[:32], width=w, label="B mean")
ax.set_title("Mean embedding (first 32 dims)")
ax.set_xlabel("latent dim")
ax.legend()

im = axes[1, 0].imshow(sim, cmap="viridis", vmin=-1, vmax=1)
axes[1, 0].set_title(f"Cosine sim ({N_VIEWS}A | {N_VIEWS}B)")
axes[1, 0].axhline(N_VIEWS - 0.5, color="red", lw=1)
axes[1, 0].axvline(N_VIEWS - 0.5, color="red", lw=1)
fig.colorbar(im, ax=axes[1, 0], fraction=0.046)

axes[1, 1].scatter(proj[:N_VIEWS, 0], proj[:N_VIEWS, 1], c="tab:blue", label="A", s=60)
axes[1, 1].scatter(proj[N_VIEWS:, 0], proj[N_VIEWS:, 1], c="tab:orange", label="B", s=60)
axes[1, 1].set_title("PCA of per-pose embeddings")
axes[1, 1].set_xlabel("PC1")
axes[1, 1].set_ylabel("PC2")
axes[1, 1].legend()

axes[1, 2].axis("off")
axes[1, 2].text(
    0.0, 0.5,
    f"Mean cosine similarity\n\n"
    f"  intra-A (across poses):  {intra_a:+.3f}\n"
    f"  intra-B (across poses):  {intra_b:+.3f}\n"
    f"  inter A vs B:            {inter:+.3f}\n\n"
    f"intra > inter  =>  the model\n"
    f"separates these two neurons\n"
    f"despite pose variation.",
    family="monospace", fontsize=11, va="center",
)

fig.suptitle("AxoNet VAE: multi-pose embeddings, two neurons", fontsize=14)
fig.tight_layout()
plt.show()

print(f"\nintra-A={intra_a:+.3f}  intra-B={intra_b:+.3f}  inter={inter:+.3f}")

## 11. Where to go from here

- **Batch over many neurons.** Wrap `render_poses` + `encode_poses` in a loop over a list of SWC files to build a `(num_neurons, 128)` matrix. See `scripts/precompute_embeddings.py` in the repo for a production version that aggregates multi-pose embeddings and saves an `.npz`.
- **Use the CLIP head.** The Stage-2 model (<https://huggingface.co/broadinstitute/axonet-clip-stage2>) projects these 128-d VAE latents into a 512-d space aligned with SciBERT text embeddings, enabling text → neuron retrieval.
- **Train your own.** See `notebooks/axonet_training_tutorial.ipynb` in this repo for the full pipeline (download → render → VAE → CLIP → eval).

### Citation

```
@misc{axonet2025,
  author = {Hall, Giles},
  title = {AxoNet: Multimodal Neuron Morphology Embeddings via 2D Projections},
  year = {2025},
  publisher = {HuggingFace},
  howpublished = {\url{https://huggingface.co/broadinstitute/axonet-vae-stage1}}
}
```